# Filament segmentation — U-Net training on Colab

Trains the runs named in `RUN_NAMES` (parameters cell; the whole experiment queue is the
`EXPERIMENTS` dict there, see `docs/PLAN.md`), writes TTA probability maps for the
held-out val split and the test set, and uploads each finished run to a **private** Kaggle
dataset `<owner>/filseg-<run>` so it can be pulled to the local machine for post-processing
tuning, ensembling and submission:

```bash
kaggle datasets download nguyenchicuong/filseg-<run> -p outputs/colab/<run> --unzip
```

Training state is written to Drive after every epoch, so a session that is cut off carries on
from the last epoch when the notebook is run again; a finished run is skipped. The run cell shows
live progress (per-epoch loss, lr, val PQ/SQ/Dice, ETA) and a queue table with the estimated finish
time; the last section plots the curves. Drive needs ~0.4 GB per run (checkpoint + resume state).

**Before running**

1. Locally: `python scripts/package_colab.py`, then upload `build/colab/filseg-colab.zip` to
   `MyDrive/filseg/`. Upload it again whenever the code changes.
2. A Colab secret **`KAGGLE_API_TOKEN`** (🔑 in the left sidebar, notebook access on) holding the
   contents of `~/.kaggle/access_token` (`cat ~/.kaggle/access_token`), not a `kaggle.json`.
   The account must have joined the competition.
3. Runtime → Change runtime type → GPU (L4 or A100; a T4 works with `--bs 4` at 1024 px).

| GPU | ResNet34, 1024 px, bs 8, 30 epochs (rough estimate) |
|---|---|
| A100 | ~1 h |
| L4 | ~2 h |
| T4 | ~4 h (bs 4) |

EfficientNet-B4 / ConvNeXt-Tiny take roughly 1.5–2x as long; 60 epochs twice as long.
Several runs fit one session on an A100: list them in `RUN_NAMES`.

In [ ]:
# colab-only
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# colab-only
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# parameters
WORK = '/content/drive/MyDrive/filseg'          # bundle + per-epoch training state live here
BUNDLE = f'{WORK}/filseg-colab.zip'
CODE = '/content/filseg'                          # where the bundle is unpacked
OUT = '/content/out'                              # probability maps, before upload
OWNER = 'nguyenchicuong'                          # Kaggle account that receives the private datasets
COMPETITION = 'filament-segmentation-2026'

# The experiment queue (docs/PLAN.md). Each entry: arguments for scripts/train_unet.py.
# --split train keeps the fixed val split held out (needed to tune post-processing and to compare);
# --split all trains on every image for the final model (no val maps are written then).
# Batch sizes are for a 24 GB GPU (L4) or larger; on a T4 (16 GB) halve them.
EXPERIMENTS = {
    # ---- first queue (2026-10-24, done): single 70/30 split, one annotator per sample
    'unet1024_r34':      '--res 1024 --bs 8 --epochs 30 --encoder resnet34',
    'unet1024_r34_grp':  '--res 1024 --bs 8 --epochs 30 --encoder resnet34 --targets group',
    'unet1024_effb4':    '--res 1024 --bs 6 --epochs 30 --encoder efficientnet-b4',
    'unet1024_cnxt':     '--res 1024 --bs 6 --epochs 30 --encoder tu-convnext_tiny --lr 5e-4',
    'unet1024_r34_e60':  '--res 1024 --bs 8 --epochs 60 --encoder resnet34',
    'unet1024_final_full': '--res 1024 --bs 8 --epochs 30 --encoder resnet34 --split all',
}
# ---- second queue (docs/PLAN.md R1-R3): 5-fold CV by image (R1), soft multi-annotator targets (R2),
# native 2048 px with 1024 px crops (R3). A fold run writes the maps of its held-out images, so the
# five folds together give out-of-fold maps for all 707 training images; the watcher assembles them,
# tunes on them and submits the mean of the five folds' test maps.
FOLD = '--targets soft --epochs 30 --val-every 10'
EXPERIMENTS['r34_s1024_f0'] = f'--res 1024 --bs 8 --encoder resnet34 {FOLD} --fold 0'           # control: soft, whole image @1024
for k in range(5):
    EXPERIMENTS[f'r34_c2048_f{k}'] = f'--res 2048 --crop 1024 --bs 8 --encoder resnet34 {FOLD} --fold {k}'
    EXPERIMENTS[f'effb4_c2048_f{k}'] = f'--res 2048 --crop 1024 --bs 6 --encoder efficientnet-b4 {FOLD} --fold {k}'
# Which of them this session trains, in order (a finished + uploaded run is skipped, a run cut
# short resumes from its last epoch on Drive). Each run is uploaded as soon as it finishes, and
# scripts/watch_colab.py on the local machine picks it up. Roughly 12-14 h on an L4: the two fold-0
# runs first (soft @1024 vs crops @2048, ~2 h together), then EfficientNet-B4 folds, then ResNet34.
RUN_NAMES = (['r34_s1024_f0', 'r34_c2048_f0'] + [f'effb4_c2048_f{k}' for k in range(5)]
             + [f'r34_c2048_f{k}' for k in range(1, 5)])
RUNS = [dict(name=n, args=EXPERIMENTS[n]) for n in RUN_NAMES]
WORKERS = 8
UPLOAD = True                                     # push each finished run to Kaggle
RELEASE_GPU = True                                # disconnect the runtime at the end

In [ ]:
# colab-only
!pip install -q -U kaggle segmentation-models-pytorch==0.5.0
import pathlib, subprocess
from google.colab import userdata
secret = userdata.get('KAGGLE_API_TOKEN').strip()
assert not secret.startswith('{'), ("KAGGLE_API_TOKEN holds a kaggle.json file; it must hold the contents of "
                                    "~/.kaggle/access_token (cat ~/.kaggle/access_token)")
token_path = pathlib.Path.home() / '.kaggle' / 'access_token'
token_path.parent.mkdir(exist_ok=True)
token_path.write_text(secret)
token_path.chmod(0o600)
check = subprocess.run(['kaggle', 'competitions', 'files', COMPETITION, '--page-size', '1'],
                       capture_output=True, text=True)
assert check.returncode == 0 and 'MAGFiLO' in check.stdout, f"Kaggle token check failed:\n{check.stdout}\n{check.stderr}"
print('Kaggle token OK')

In [ ]:
# colab-only
import os
assert os.path.isfile(BUNDLE), f"{BUNDLE} is missing: run scripts/package_colab.py locally and upload the zip"
!rm -rf "$CODE" && mkdir -p "$CODE" && unzip -q "$BUNDLE" -d "$CODE"
os.chdir(CODE)
missing = [f for f in ('scripts/train_unet.py', 'scripts/infer_probs.py', 'scripts/kaggle_upload.py',
                       'src/filseg/dl.py') if not os.path.exists(f)]
assert not missing, f"bundle is missing {missing}: it is an old zip, upload the current one"
print('code in', CODE)

## Data

The competition files (~720 MB) download in a minute or two and unpack to `data/` inside the code
directory, the layout the scripts expect.

In [ ]:
import os, pathlib, subprocess, zipfile
data = pathlib.Path(CODE) / 'data'
root = data / 'MAGFiLO_1.0_Kaggle_2026'
if not root.exists():
    data.mkdir(exist_ok=True)
    subprocess.run(['kaggle', 'competitions', 'download', COMPETITION, '-p', str(data), '-q'], check=True)
    for z in data.glob('*.zip'):
        zipfile.ZipFile(z).extractall(data)
        z.unlink()
n_train = len(list((root / 'train' / 'train_images').glob('*.jpeg')))
n_test = len(list((root / 'test' / 'test_images').glob('*.jpeg')))
assert (n_train, n_test) == (707, 180), f"unexpected data: {n_train} train / {n_test} test images"
print(f'{n_train} train / {n_test} test images')

## Train, predict, upload

Each run trains with `--resume` against its state on Drive, then writes 8x-TTA probability maps
(uint8, at the training resolution) for the val split and the test set, and uploads checkpoint +
maps to Kaggle, where `scripts/watch_colab.py` picks it up and submits it. A run that fails is
reported and the next one still runs.

**What the output shows**

- before the queue and after every run: a status table — each run's state, epochs done, best val
  PQ, time used, and the estimated time to finish the run and the whole queue;
- while a run trains, 4 progress lines per epoch (loss, it/s, time left in the epoch), then an epoch
  line: `loss`, `lr`, epoch time, and every 5 epochs **val PQ / SQ / pixel Dice / TP / FP / FN**
  (`* best` marks a new best checkpoint), elapsed time, run ETA and its clock time (UTC);
- every 5 epochs a queue ETA line.

Everything printed is also written to `MyDrive/filseg/runs/<run>.log` (readable from anywhere,
also after the session ends), and each epoch's numbers to `<run>.history.csv` beside it.

In [ ]:
import json, shutil, subprocess, sys, time
from datetime import datetime, timedelta, timezone
from pathlib import Path

MODELS = Path(WORK) / 'runs'
MODELS.mkdir(parents=True, exist_ok=True)
# Relative cost of one epoch against ResNet34 at the same resolution (rough; refined by measurement).
COST = {'resnet34': 1.0, 'efficientnet-b4': 1.8, 'tu-convnext_tiny': 1.8}
INFER_MIN = 6           # TTA maps for val + test, minutes on an L4 (rough)
ref_sec_per_epoch = None  # measured seconds/epoch of a ResNet34 run, filled in as runs train

def fmt(sec):
    sec = int(max(sec or 0, 0))
    return f'{sec // 3600}h{sec % 3600 // 60:02d}m'

def arg(args, key, default=None):
    a = args.split()
    return a[a.index(key) + 1] if key in a else default

def run_status(run):
    name = run['name']
    hist = MODELS / f'{name}.history.csv'
    epochs = int(arg(run['args'], '--epochs', 30))
    rows = [l.split(',') for l in hist.read_text().splitlines()[1:]] if hist.exists() else []
    done = int(rows[-1][0]) if rows else 0
    used = float(rows[-1][5]) if rows else 0.0
    pqs = [float(r[6]) for r in rows if len(r) > 6 and r[6]]
    sec_ep = used / done if done else None
    cost = COST.get(arg(run['args'], '--encoder', 'resnet34'), 1.5)
    if sec_ep is None and ref_sec_per_epoch:
        sec_ep = ref_sec_per_epoch * cost
    if (MODELS / f'{name}.uploaded').exists():
        state, left = 'uploaded', 0
    else:
        state = 'training' if 0 < done < epochs else ('trained' if done >= epochs else 'queued')
        maps_done = (Path(OUT) / name / 'probs_test').is_dir()
        left = (sec_ep * (epochs - done) if sec_ep else None)
        left = None if left is None else left + (0 if maps_done else INFER_MIN * 60)
    return dict(run=name, state=state, epochs=f'{done}/{epochs}', best_pq=f'{max(pqs):.4f}' if pqs else '-',
                used=fmt(used), left='?' if left is None else fmt(left), left_s=left)

def show_queue(title):
    global ref_sec_per_epoch
    if ref_sec_per_epoch is None:  # seed the estimate from any ResNet34 run already on Drive
        for r in RUNS:
            h = MODELS / f"{r['name']}.history.csv"
            if (arg(r['args'], '--encoder', 'resnet34') == 'resnet34' and arg(r['args'], '--res') == '1024'
                    and h.exists() and len(h.read_text().splitlines()) > 1):
                last = h.read_text().splitlines()[-1].split(',')
                ref_sec_per_epoch = float(last[5]) / int(last[0])
                break
    rows = [run_status(r) for r in RUNS]
    known = [r['left_s'] for r in rows if r['left_s'] is not None]
    unknown = sum(r['left_s'] is None for r in rows)
    total = sum(known)
    eta = datetime.now(timezone.utc) + timedelta(seconds=total)
    print(f'\n===== {title} — {datetime.now(timezone.utc):%H:%M} UTC =====')
    print(f"{'run':22s} {'state':9s} {'epochs':7s} {'best pq':8s} {'used':7s} {'left':7s}")
    for r in rows:
        print(f"{r['run']:22s} {r['state']:9s} {r['epochs']:7s} {r['best_pq']:8s} {r['used']:7s} {r['left']:7s}")
    tail = f' + {unknown} run(s) not yet estimable' if unknown else ''
    print(f'queue: ~{fmt(total)} left{tail} -> done ~{eta:%Y-%m-%d %H:%M} UTC\n', flush=True)

def stream(cmd, log_path, on_line=None):
    # Run a command, show its output live in this cell and append it to a log file on Drive.
    print('>>', ' '.join(map(str, cmd)), flush=True)
    with open(log_path, 'a') as log:
        proc = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line); log.flush()
            if line.startswith('@@ '):
                if on_line: on_line(json.loads(line[3:]))
                continue
            if 'Warning' in line or 'warnings.warn' in line:
                continue
            print(line, end='', flush=True)
        return proc.wait()

def on_progress(run):
    def cb(p):
        global ref_sec_per_epoch
        if (arg(run['args'], '--encoder', 'resnet34') == 'resnet34' and arg(run['args'], '--res') == '1024'
                and arg(run['args'], '--targets', 'mask') == 'mask'):
            ref_sec_per_epoch = p['sec_per_epoch']
        if p['epoch'] % 5 == 0 and p['epoch'] < p['epochs']:
            show_queue(f"queue after {run['name']} epoch {p['epoch']}")
    return cb

show_queue('queue at start')
failed = []
for i, run in enumerate(RUNS, 1):
    name, args = run['name'], run['args'].split()
    out = Path(OUT) / name
    done_marker = MODELS / f'{name}.uploaded'
    log_path = MODELS / f'{name}.log'
    if done_marker.exists():
        print(f'[{i}/{len(RUNS)}] {name}: already trained and uploaded, skipping (delete {done_marker} to redo)')
        continue
    print(f'\n########## [{i}/{len(RUNS)}] {name}: {run["args"]} ##########', flush=True)
    t0 = time.time()
    code_ = stream([sys.executable, '-u', 'scripts/train_unet.py', *args, '--name', name, '--out-dir', MODELS,
                    '--workers', WORKERS, '--resume'], log_path, on_progress(run))
    ckpt = MODELS / f'{name}.pt'
    if code_ != 0 or not ckpt.exists():
        failed.append(name)
        print(f'{name}: TRAINING FAILED (exit {code_}), last lines in {log_path}', flush=True)
        continue
    print(f'{name}: trained in {fmt(time.time() - t0)}', flush=True)
    t1 = time.time()
    shutil.rmtree(out, ignore_errors=True)
    out.mkdir(parents=True)
    for f in (f'{name}.pt', f'{name}.json', f'{name}.history.csv'):
        shutil.copy(MODELS / f, out / f)
    ok = stream([sys.executable, '-u', 'scripts/infer_probs.py', '--ckpt', ckpt, '--which', 'test', '--tta',
                 '--out', out / 'probs_test'], log_path) == 0
    if '--fold' in args:  # the fold's held-out images: its share of the out-of-fold maps
        ok &= stream([sys.executable, '-u', 'scripts/infer_probs.py', '--ckpt', ckpt, '--which', 'fold', '--tta',
                      '--out', out / 'probs_val'], log_path) == 0
    elif '--split' not in args or args[args.index('--split') + 1] == 'train':
        ok &= stream([sys.executable, '-u', 'scripts/infer_probs.py', '--ckpt', ckpt, '--which', 'val', '--tta',
                      '--out', out / 'probs_val'], log_path) == 0
    print(f'{name}: TTA maps in {fmt(time.time() - t1)}', flush=True)
    if not ok:
        failed.append(name)
        continue
    if UPLOAD:
        slug = 'filseg-' + name.replace('_', '-')
        meta = json.loads((out / f'{name}.json').read_text())
        if stream([sys.executable, '-u', 'scripts/kaggle_upload.py', '--dir', out, '--owner', OWNER, '--slug', slug,
                   '--message', f"best val pq {meta.get('best_val_pq')}"], log_path) != 0:
            failed.append(name + ' (upload)')
        else:
            done_marker.write_text(slug)
            print(f'{name}: uploaded to kaggle.com/datasets/{OWNER}/{slug}', flush=True)
    show_queue(f'queue after {name}')
print('failed:', failed or 'none')

## Results and curves

The status of every run, the best val PQ of each, and its loss / val PQ / val Dice curves (from the
`history.csv` files on Drive). This cell also works on its own after a reconnect: run the parameters
cell, the Drive cell and this one. Paste the table back into the chat.

In [ ]:
import csv
import matplotlib.pyplot as plt
from pathlib import Path

MODELS = Path(WORK) / 'runs'
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
print(f"{'run':22s} {'epochs':>6s} {'best pq':>8s} {'last dice':>9s} {'hours':>6s}  uploaded")
for name in RUN_NAMES:
    hist = MODELS / f'{name}.history.csv'
    if not hist.exists():
        print(f'{name:22s} {"-":>6s}')
        continue
    rows = list(csv.DictReader(open(hist)))
    ep = [int(r['epoch']) for r in rows]
    ax[0].plot(ep, [float(r['loss']) for r in rows], '.-', label=name)
    val = [r for r in rows if r['val_pq']]
    ax[1].plot([int(r['epoch']) for r in val], [float(r['val_pq']) for r in val], 'o-', label=name)
    ax[2].plot([int(r['epoch']) for r in val], [float(r['val_dice']) for r in val], 'o-', label=name)
    best = max((float(r['val_pq']) for r in val), default=float('nan'))
    dice = float(val[-1]['val_dice']) if val else float('nan')
    hours = float(rows[-1]['elapsed_s']) / 3600
    print(f"{name:22s} {ep[-1]:6d} {best:8.4f} {dice:9.4f} {hours:6.2f}  {(MODELS / f'{name}.uploaded').exists()}")
for a, t in zip(ax, ['train loss', 'val PQ (cc post, no TTA)', 'val pixel Dice']):
    a.set_title(t); a.set_xlabel('epoch'); a.grid(alpha=0.3)
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## Release the GPU

In [ ]:
# colab-only
# Reached only when every cell above ran; after an error the runtime stays up for a look.
if RELEASE_GPU:
    from google.colab import runtime
    runtime.unassign()